# LSTM with Attention — Multivariate Time-Series Forecasting

## Objective

In this stage, an LSTM model with an Attention mechanism is developed for next-day sales forecasting.

The model uses the previous 28 days of historical information and 22 engineered features to predict the next day's sales.

## Model Architecture

```text
28 Days × 22 Features
          ↓
      LSTM Layer
          ↓
   Attention Layer
          ↓
      Dense Layer
          ↓
     Output Layer
          ↓
   Next-Day Sales

In [1]:
import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import tensorflow as tf

from sklearn.preprocessing import StandardScaler

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.20.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [2]:
# Project configuration

SEQUENCE_LENGTH = 28
FORECAST_HORIZON = 1
TARGET_COLUMN = "sales"

FEATURE_COLUMNS = [
    "sales",
    "sell_price",
    "price_available",
    "day_of_month",
    "week_of_year",
    "day_of_year",
    "quarter",
    "is_weekend",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28",
    "price_change_1",
    "price_change_pct_1",
    "price_relative_7",
    "is_event_day",
    "event_count",
    "snap_active"
]

DATA_PATH = "/kaggle/input/datasets/gou14226/m5-features-event-snap/features_event_snap.parquet"

TRAIN_END_DATE = "2016-03-27"
VALIDATION_START_DATE = "2016-03-28"
VALIDATION_END_DATE = "2016-04-24"

BATCH_SIZE = 64
EPOCHS = 10
TRAIN_STEPS = 5000
VALIDATION_STEPS = 500

print("Configuration loaded successfully.")
print("Sequence length:", SEQUENCE_LENGTH)
print("Forecast horizon:", FORECAST_HORIZON)
print("Number of features:", len(FEATURE_COLUMNS))
print("Batch size:", BATCH_SIZE)

Configuration loaded successfully.
Sequence length: 28
Forecast horizon: 1
Number of features: 22
Batch size: 64


In [4]:
# Check the Parquet dataset

parquet_file = pq.ParquetFile(DATA_PATH)

print("Dataset exists:", os.path.exists(DATA_PATH))
print("Number of rows:", parquet_file.metadata.num_rows)
print("Number of columns:", parquet_file.metadata.num_columns)
print("Number of row groups:", parquet_file.num_row_groups)

print("\nDataset columns:")
print(parquet_file.schema.names)

Dataset exists: True
Number of rows: 58327370
Number of columns: 42
Number of row groups: 61

Dataset columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'price_available', 'day_of_month', 'week_of_year', 'day_of_year', 'quarter', 'is_weekend', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_28', 'rolling_std_7', 'rolling_std_28', 'price_change_1', 'price_change_pct_1', 'price_relative_7', 'is_event_day', 'event_count', 'snap_active']


In [5]:
# Verify date range and train-validation split

first_row_group = parquet_file.read_row_group(0).to_pandas()
last_row_group = parquet_file.read_row_group(
    parquet_file.num_row_groups - 1
).to_pandas()

first_date = pd.to_datetime(first_row_group["date"]).min()
last_date = pd.to_datetime(last_row_group["date"]).max()

print("Global date range:")
print("Start date:", first_date.date())
print("End date  :", last_date.date())

print("\nTrain / Validation split:")
print("Train end date      :", TRAIN_END_DATE)
print("Validation start    :", VALIDATION_START_DATE)
print("Validation end      :", VALIDATION_END_DATE)

Global date range:
Start date: 2011-01-29
End date  : 2016-04-24

Train / Validation split:
Train end date      : 2016-03-27
Validation start    : 2016-03-28
Validation end      : 2016-04-24


In [6]:
# Feature preparation function

def prepare_features(df, feature_columns):
    df = df.copy()

    price_features = [
        "sell_price",
        "price_change_1",
        "price_change_pct_1",
        "price_relative_7"
    ]

    history_features = [
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
        "rolling_mean_7",
        "rolling_mean_28",
        "rolling_std_7",
        "rolling_std_28"
    ]

    # Fill missing price-related features with 0
    for col in price_features:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # Fill missing historical/rolling features with 0
    for col in history_features:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    X = df[feature_columns].to_numpy(dtype=np.float64)

    return X

print("Feature preparation function created successfully.")

Feature preparation function created successfully.


In [7]:
# Fit StandardScaler using training data only

scaler = StandardScaler()

train_rows = 0

for rg_idx in range(parquet_file.num_row_groups):
    df_rg = parquet_file.read_row_group(rg_idx).to_pandas()

    df_rg["date"] = pd.to_datetime(df_rg["date"])

    train_df = df_rg[
        df_rg["date"] <= TRAIN_END_DATE
    ]

    if len(train_df) > 0:
        X_train = prepare_features(
            train_df,
            FEATURE_COLUMNS
        )

        scaler.partial_fit(X_train)

        train_rows += len(train_df)

    del df_rg
    del train_df

print("Scaler fitted successfully.")
print("Training rows used:", train_rows)
print("Number of features:", len(FEATURE_COLUMNS))

print("\nFirst 5 feature means:")
print(scaler.mean_[:5])

print("\nFirst 5 feature standard deviations:")
print(scaler.scale_[:5])

Scaler fitted successfully.
Training rows used: 57473650
Number of features: 22

First 5 feature means:
[ 1.12245843  3.4636664   0.7859991  15.71458886 26.0397878 ]

First 5 feature standard deviations:
[ 3.87698193  3.51547202  0.41012744  8.79354866 15.17225613]


In [8]:
# Create sequences from a single time-series

def create_sequences(
    df,
    feature_columns,
    target_column,
    scaler,
    sequence_length=28
):
    df = df.sort_values("date").reset_index(drop=True)

    X_raw = prepare_features(
        df,
        feature_columns
    )

    X_scaled = scaler.transform(X_raw)

    y = df[target_column].to_numpy(
        dtype=np.float32
    )

    X_sequences = []
    y_sequences = []

    for i in range(sequence_length, len(df)):
        X_sequences.append(
            X_scaled[i-sequence_length:i]
        )

        y_sequences.append(
            y[i]
        )

    if len(X_sequences) == 0:
        return (
            np.empty(
                (
                    0,
                    sequence_length,
                    len(feature_columns)
                ),
                dtype=np.float32
            ),
            np.empty(
                (0,),
                dtype=np.float32
            )
        )

    return (
        np.asarray(
            X_sequences,
            dtype=np.float32
        ),
        np.asarray(
            y_sequences,
            dtype=np.float32
        )
    )

print("Sequence generation function created successfully.")

Sequence generation function created successfully.


In [9]:
# Test sequence generation on one series

test_item = "HOBBIES_1_001"
test_store = "CA_1"

test_rows = []

for rg_idx in range(parquet_file.num_row_groups):
    df_rg = parquet_file.read_row_group(rg_idx).to_pandas()

    mask = (
        (df_rg["item_id"] == test_item) &
        (df_rg["store_id"] == test_store)
    )

    if mask.any():
        test_rows.append(
            df_rg.loc[mask]
        )

    del df_rg

test_series = pd.concat(
    test_rows,
    ignore_index=True
)

test_series["date"] = pd.to_datetime(
    test_series["date"]
)

test_series = test_series.sort_values(
    "date"
).reset_index(drop=True)

X_test, y_test = create_sequences(
    test_series,
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    scaler,
    SEQUENCE_LENGTH
)

print("Test series:", test_item, "/", test_store)
print("Number of rows:", len(test_series))
print(
    "Date range:",
    test_series["date"].min().date(),
    "to",
    test_series["date"].max().date()
)

print("\nSequence shapes:")
print("X shape:", X_test.shape)
print("y shape:", y_test.shape)

print("\nData quality:")
print("NaN in X:", np.isnan(X_test).sum())
print("Inf in X:", np.isinf(X_test).sum())
print("NaN in y:", np.isnan(y_test).sum())
print("Inf in y:", np.isinf(y_test).sum())

print("\nFirst 10 targets:")
print(y_test[:10])

Test series: HOBBIES_1_001 / CA_1
Number of rows: 1913
Date range: 2011-01-29 to 2016-04-24

Sequence shapes:
X shape: (1885, 28, 22)
y shape: (1885,)

Data quality:
NaN in X: 0
Inf in X: 0
NaN in y: 0
Inf in y: 0

First 10 targets:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [10]:
# Memory-efficient streaming batch generator

class StreamingBatchGenerator:

    def __init__(
        self,
        parquet_path,
        feature_columns,
        target_column,
        scaler,
        sequence_length,
        start_date,
        end_date,
        batch_size=64
    ):
        self.parquet_path = parquet_path
        self.feature_columns = feature_columns
        self.target_column = target_column
        self.scaler = scaler
        self.sequence_length = sequence_length
        self.start_date = pd.to_datetime(start_date)
        self.end_date = pd.to_datetime(end_date)
        self.batch_size = batch_size

    def batches(self):

        pf = pq.ParquetFile(self.parquet_path)

        history_buffer = {}

        X_batch = []
        y_batch = []

        for rg_idx in range(pf.num_row_groups):

            table = pf.read_row_group(rg_idx)
            df_rg = table.to_pandas()

            df_rg["date"] = pd.to_datetime(df_rg["date"])

            df_rg = df_rg.sort_values(
                ["item_id", "store_id", "date"]
            )

            for (item_id, store_id), group in df_rg.groupby(
                ["item_id", "store_id"],
                sort=False
            ):

                key = (item_id, store_id)

                group = group.reset_index(drop=True)

                # Add previous 28 days when the series
                # continues from an earlier row group
                if key in history_buffer:

                    previous_history = history_buffer[key]

                    group = pd.concat(
                        [previous_history, group],
                        ignore_index=True
                    )

                # Prepare and scale features
                X_raw = prepare_features(
                    group,
                    self.feature_columns
                )

                X_scaled = self.scaler.transform(
                    X_raw
                )

                y = group[
                    self.target_column
                ].to_numpy(dtype=np.float32)

                dates = group["date"].to_numpy()

                # Create sequences
                for i in range(
                    self.sequence_length,
                    len(group)
                ):

                    target_date = pd.Timestamp(
                        dates[i]
                    )

                    if (
                        self.start_date
                        <= target_date
                        <= self.end_date
                    ):

                        X_batch.append(
                            X_scaled[
                                i-self.sequence_length:i
                            ]
                        )

                        y_batch.append(
                            y[i]
                        )

                        if len(X_batch) == self.batch_size:

                            yield (
                                np.asarray(
                                    X_batch,
                                    dtype=np.float32
                                ),
                                np.asarray(
                                    y_batch,
                                    dtype=np.float32
                                )
                            )

                            X_batch = []
                            y_batch = []

                # Keep last 28 days for the next row group
                history_buffer[key] = group.tail(
                    self.sequence_length
                ).copy()

            del df_rg
            del table

        # Yield final partial batch
        if len(X_batch) > 0:

            yield (
                np.asarray(
                    X_batch,
                    dtype=np.float32
                ),
                np.asarray(
                    y_batch,
                    dtype=np.float32
                )
            )


print("Streaming batch generator created successfully.")

Streaming batch generator created successfully.


In [11]:
# Test one training batch from the streaming generator

train_generator = StreamingBatchGenerator(
    parquet_path=DATA_PATH,
    feature_columns=FEATURE_COLUMNS,
    target_column=TARGET_COLUMN,
    scaler=scaler,
    sequence_length=SEQUENCE_LENGTH,
    start_date="2011-01-29",
    end_date=TRAIN_END_DATE,
    batch_size=BATCH_SIZE
)

X_train_test, y_train_test = next(
    train_generator.batches()
)

print("Training batch:")
print("X shape:", X_train_test.shape)
print("y shape:", y_train_test.shape)

print("\nDtypes:")
print("X dtype:", X_train_test.dtype)
print("y dtype:", y_train_test.dtype)

print("\nData quality:")
print(
    "NaN in X:",
    np.isnan(X_train_test).sum()
)

print(
    "Inf in X:",
    np.isinf(X_train_test).sum()
)

print(
    "NaN in y:",
    np.isnan(y_train_test).sum()
)

print(
    "Inf in y:",
    np.isinf(y_train_test).sum()
)

print("\nFirst 10 targets:")
print(y_train_test[:10])

Training batch:
X shape: (64, 28, 22)
y shape: (64,)

Dtypes:
X dtype: float32
y dtype: float32

Data quality:
NaN in X: 0
Inf in X: 0
NaN in y: 0
Inf in y: 0

First 10 targets:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [12]:
# Test one validation batch from the streaming generator

validation_generator = StreamingBatchGenerator(
    parquet_path=DATA_PATH,
    feature_columns=FEATURE_COLUMNS,
    target_column=TARGET_COLUMN,
    scaler=scaler,
    sequence_length=SEQUENCE_LENGTH,
    start_date=VALIDATION_START_DATE,
    end_date=VALIDATION_END_DATE,
    batch_size=BATCH_SIZE
)

X_val_test, y_val_test = next(
    validation_generator.batches()
)

print("Validation batch:")
print("X shape:", X_val_test.shape)
print("y shape:", y_val_test.shape)

print("\nDtypes:")
print("X dtype:", X_val_test.dtype)
print("y dtype:", y_val_test.dtype)

print("\nData quality:")
print(
    "NaN in X:",
    np.isnan(X_val_test).sum()
)

print(
    "Inf in X:",
    np.isinf(X_val_test).sum()
)

print(
    "NaN in y:",
    np.isnan(y_val_test).sum()
)

print(
    "Inf in y:",
    np.isinf(y_val_test).sum()
)

print("\nFirst 10 validation targets:")
print(y_val_test[:10])

Validation batch:
X shape: (64, 28, 22)
y shape: (64,)

Dtypes:
X dtype: float32
y dtype: float32

Data quality:
NaN in X: 0
Inf in X: 0
NaN in y: 0
Inf in y: 0

First 10 validation targets:
[1. 0. 0. 0. 0. 0. 1. 0. 4. 2.]


In [13]:
# Create TensorFlow datasets from the streaming generators

def train_batch_stream():
    generator = StreamingBatchGenerator(
        parquet_path=DATA_PATH,
        feature_columns=FEATURE_COLUMNS,
        target_column=TARGET_COLUMN,
        scaler=scaler,
        sequence_length=SEQUENCE_LENGTH,
        start_date="2011-01-29",
        end_date=TRAIN_END_DATE,
        batch_size=BATCH_SIZE
    )

    yield from generator.batches()


def validation_batch_stream():
    generator = StreamingBatchGenerator(
        parquet_path=DATA_PATH,
        feature_columns=FEATURE_COLUMNS,
        target_column=TARGET_COLUMN,
        scaler=scaler,
        sequence_length=SEQUENCE_LENGTH,
        start_date=VALIDATION_START_DATE,
        end_date=VALIDATION_END_DATE,
        batch_size=BATCH_SIZE
    )

    yield from generator.batches()


output_signature = (
    tf.TensorSpec(
        shape=(None, SEQUENCE_LENGTH, len(FEATURE_COLUMNS)),
        dtype=tf.float32
    ),
    tf.TensorSpec(
        shape=(None,),
        dtype=tf.float32
    )
)

train_dataset = tf.data.Dataset.from_generator(
    train_batch_stream,
    output_signature=output_signature
)

validation_dataset = tf.data.Dataset.from_generator(
    validation_batch_stream,
    output_signature=output_signature
)

print("TensorFlow datasets created successfully.")

TensorFlow datasets created successfully.


I0000 00:00:1789543577.985669      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789543577.988989      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [14]:
# Verify one batch from each TensorFlow dataset

X_tf_train, y_tf_train = next(iter(train_dataset))
X_tf_val, y_tf_val = next(iter(validation_dataset))

print("Training batch:")
print("X shape:", X_tf_train.shape)
print("y shape:", y_tf_train.shape)

print("\nValidation batch:")
print("X shape:", X_tf_val.shape)
print("y shape:", y_tf_val.shape)

print("\nDtypes:")
print("Training X:", X_tf_train.dtype)
print("Training y:", y_tf_train.dtype)
print("Validation X:", X_tf_val.dtype)
print("Validation y:", y_tf_val.dtype)

print("\nData quality:")
print(
    "Training X NaN:",
    tf.reduce_sum(
        tf.cast(tf.math.is_nan(X_tf_train), tf.int32)
    ).numpy()
)

print(
    "Validation X NaN:",
    tf.reduce_sum(
        tf.cast(tf.math.is_nan(X_tf_val), tf.int32)
    ).numpy()
)

print(
    "Training y NaN:",
    tf.reduce_sum(
        tf.cast(tf.math.is_nan(y_tf_train), tf.int32)
    ).numpy()
)

print(
    "Validation y NaN:",
    tf.reduce_sum(
        tf.cast(tf.math.is_nan(y_tf_val), tf.int32)
    ).numpy()
)

print("\nFirst 10 validation targets:")
print(y_tf_val[:10].numpy())

Training batch:
X shape: (64, 28, 22)
y shape: (64,)

Validation batch:
X shape: (64, 28, 22)
y shape: (64,)

Dtypes:
Training X: <dtype: 'float32'>
Training y: <dtype: 'float32'>
Validation X: <dtype: 'float32'>
Validation y: <dtype: 'float32'>

Data quality:
Training X NaN: 0
Validation X NaN: 0
Training y NaN: 0
Validation y NaN: 0

First 10 validation targets:
[1. 0. 0. 0. 0. 0. 1. 0. 4. 2.]


In [15]:
# LSTM + Attention model

inputs = tf.keras.Input(
    shape=(SEQUENCE_LENGTH, len(FEATURE_COLUMNS))
)

# LSTM must return the output of every time step
lstm_output = tf.keras.layers.LSTM(
    64,
    activation="tanh",
    return_sequences=True
)(inputs)

# Attention over the 28 time steps
attention_output = tf.keras.layers.Attention()(
    [lstm_output, lstm_output]
)

# Convert sequence representation into a single vector
context_vector = tf.keras.layers.GlobalAveragePooling1D()(
    attention_output
)

dense_output = tf.keras.layers.Dense(
    32,
    activation="relu"
)(context_vector)

outputs = tf.keras.layers.Dense(1)(
    dense_output
)

model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 28, 22)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 28, 64)    │     22,272 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 28, 64)    │          0 │ lstm[0][0],       │
│ (Attention)         │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ attention[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │      2,080 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │         33 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,385 (95.25 KB)

 Trainable params: 24,385 (95.25 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
# Test model forward pass on one training batch

sample_predictions = model(
    X_tf_train,
    training=False
)

print("Model output shape:", sample_predictions.shape)
print("Model output dtype:", sample_predictions.dtype)

print("\nData quality:")
print(
    "NaN in predictions:",
    tf.reduce_sum(
        tf.cast(
            tf.math.is_nan(sample_predictions),
            tf.int32
        )
    ).numpy()
)

print(
    "Inf in predictions:",
    tf.reduce_sum(
        tf.cast(
            tf.math.is_inf(sample_predictions),
            tf.int32
        )
    ).numpy()
)

print("\nFirst 10 predictions:")
print(
    sample_predictions[:10].numpy().reshape(-1)
)

Model output shape: (64, 1)
Model output dtype: <dtype: 'float32'>

Data quality:
NaN in predictions: 0
Inf in predictions: 0

First 10 predictions:
[-0.46653947 -0.4649122  -0.46390632 -0.46424368 -0.46382403 -0.46325597
 -0.46271223 -0.46251214 -0.4589636  -0.46557686]


In [17]:
# Early stopping configuration

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

print("Training configuration:")
print("Epochs:", EPOCHS)
print("Training steps:", TRAIN_STEPS)
print("Validation steps:", VALIDATION_STEPS)
print("Early stopping patience:", 2)
print("Restore best weights:", True)

Training configuration:
Epochs: 10
Training steps: 5000
Validation steps: 500
Early stopping patience: 2
Restore best weights: True


In [18]:
# Train LSTM + Attention model

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    steps_per_epoch=TRAIN_STEPS,
    validation_steps=VALIDATION_STEPS,
    epochs=EPOCHS,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 58s 11ms/step - loss: 6.7484 - mae: 0.8790 - val_loss: 4.1088 - val_mae: 1.0209
Epoch 2/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 55s 11ms/step - loss: 3.5885 - mae: 0.5697 - val_loss: 4.1286 - val_mae: 0.9967
Epoch 3/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 56s 11ms/step - loss: 1.7172 - mae: 0.6629 - val_loss: 3.9109 - val_mae: 1.0736
Epoch 4/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 59s 12ms/step - loss: 4.6901 - mae: 0.9456 - val_loss: 3.9949 - val_mae: 1.0081
Epoch 5/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 57s 11ms/step - loss: 3.4041 - mae: 0.8070 - val_loss: 3.8518 - val_mae: 1.0152
Epoch 6/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 57s 11ms/step - loss: 3.2230 - mae: 0.9940 - val_loss: 3.8829 - val_mae: 0.9870
Epoch 7/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 54s 11ms/step - loss: 3.2248 - mae: 0.8376 - val_loss: 3.8129 - val_mae: 0.9439
Epoch 8/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 56s 11ms/step - loss: 0.9197 - mae: 0.4649 - val_loss: 3.7536 - val_mae: 0.9593
Epoch 9/10
5000/5000 ━━━

In [19]:
# Generate predictions for the complete validation period

val_predictions = model.predict(
    validation_dataset,
    verbose=1
).reshape(-1)

print("Prediction shape:", val_predictions.shape)

print("Expected predictions:", 853720)

print("NaN predictions:", np.isnan(val_predictions).sum())
print("Inf predictions:", np.isinf(val_predictions).sum())

print("\nFirst 10 predictions:")
print(val_predictions[:10])

13340/13340 ━━━━━━━━━━━━━━━━━━━━ 427s 31ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


Prediction shape: (853720,)
Expected predictions: 853720
NaN predictions: 0
Inf predictions: 0

First 10 predictions:
[0.651006   0.6572343  0.63044924 0.68715346 0.90737534 1.0362501
 0.96919405 0.5784578  0.5408917  0.582484  ]


In [20]:
# Collect actual validation targets
val_actuals = []

for _, y_batch in validation_dataset:
    val_actuals.append(y_batch.numpy())

val_actuals = np.concatenate(val_actuals).reshape(-1)

print("Actual shape:", val_actuals.shape)
print("Expected actuals:", 853720)

print("NaN actuals:", np.isnan(val_actuals).sum())
print("Inf actuals:", np.isinf(val_actuals).sum())

print("\nFirst 10 actual values:")
print(val_actuals[:10])

Actual shape: (853720,)
Expected actuals: 853720
NaN actuals: 0
Inf actuals: 0

First 10 actual values:
[1. 0. 0. 0. 0. 0. 1. 0. 4. 2.]


In [21]:
# Calculate validation metrics

mae = np.mean(np.abs(val_actuals - val_predictions))

rmse = np.sqrt(
    np.mean((val_actuals - val_predictions) ** 2)
)

wape = (
    np.sum(np.abs(val_actuals - val_predictions))
    / np.sum(np.abs(val_actuals))
) * 100

print("LSTM + Attention Validation Results")
print("=" * 45)

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"WAPE : {wape:.4f}%")

LSTM + Attention Validation Results
MAE  : 1.0058
RMSE : 2.2867
WAPE : 72.5478%


In [22]:
# Complete model comparison

comparison = pd.DataFrame({
    "Model": [
        "Naive",
        "Seasonal Naive",
        "28-day Moving Average",
        "RNN",
        "LSTM",
        "LSTM + Attention"
    ],
    "MAE": [
        1.1798,
        1.2054,
        1.0050,
        1.0326,
        1.0220,
        mae
    ],
    "RMSE": [
        2.5948,
        2.6601,
        2.0935,
        2.4927,
        2.3766,
        rmse
    ],
    "WAPE (%)": [
        85.0972,
        86.9390,
        72.4875,
        74.4774,
        73.7131,
        wape
    ]
})

comparison

,Model,MAE,RMSE,WAPE (%)
0,Naive,1.179800,2.594800,85.097200
1,Seasonal Naive,1.205400,2.660100,86.939000
2,28-day Moving Average,1.005000,2.093500,72.487500
3,RNN,1.032600,2.492700,74.477400
4,LSTM,1.022000,2.376600,73.713100
5,LSTM + Attention,1.005827,2.286688,72.547821


In [23]:
# Training summary

best_epoch = np.argmin(history.history["val_loss"]) + 1
best_val_loss = np.min(history.history["val_loss"])
best_val_mae = history.history["val_mae"][best_epoch - 1]

print("LSTM + Attention Training Summary")
print("=" * 45)
print(f"Total epochs completed : {len(history.history['loss'])}")
print(f"Best epoch             : {best_epoch}")
print(f"Best validation loss   : {best_val_loss:.4f}")
print(f"Validation MAE at best epoch : {best_val_mae:.4f}")
print(f"Early stopping patience: {early_stopping.patience}")
print(f"Restore best weights   : {early_stopping.restore_best_weights}")

LSTM + Attention Training Summary
Total epochs completed : 10
Best epoch             : 8
Best validation loss   : 3.7536
Validation MAE at best epoch : 0.9593
Early stopping patience: 2
Restore best weights   : True


# LSTM + Attention — Final Results

## Objective

An LSTM model with an Attention mechanism was developed for next-day sales forecasting using the previous 28 days of historical information and 22 engineered features.

The purpose of this experiment was to investigate whether Attention could improve forecasting performance by allowing the model to assign different importance to historical time steps.

---

## Model Configuration

- Model: LSTM + Attention
- Sequence Length: 28 days
- Forecast Horizon: 1 day
- Input Features: 22
- LSTM Units: 64
- Dense Units: 32
- Optimizer: Adam
- Learning Rate: 0.001
- Loss Function: MSE
- Training Batch Size: 64
- Maximum Epochs: 10
- Training Steps per Epoch: 5000
- Validation Steps per Epoch: 500
- Early Stopping Patience: 2
- Restore Best Weights: True

### Architecture

```text
28 Days × 22 Features
          ↓
    LSTM (64 units)
          ↓
      Attention
          ↓
Global Average Pooling
          ↓
    Dense (32 units)
          ↓
      Dense (1)
          ↓
   Next-Day Sales